In [1]:
# Importaciones
import pandas as pd
from prophet import Prophet
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Cargar datos
df = pd.read_csv('analytics_data.csv')
print(df.shape)
print(df.columns.tolist())
df.head()

(104034, 6)
['customer_id', 'order_id', 'order_date', 'order_value', 'state', 'city']


,customer_id,order_id,order_date,order_value,state,city
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27.000,141.90,SP,cajamar
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27.000,27.19,SP,osasco
2,0004bd2a26a76fe21f786e4fbd80607f,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16.000,166.98,SP,sao paulo
3,00053a61a98854899e70ed204dd4bafe,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41.000,419.18,PR,curitiba
4,0005e1862207bf6ccc02e4228effd9a0,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12.000,150.12,RJ,teresopolis


In [2]:
# Preparar datos para Prophet
df['order_date'] = pd.to_datetime(df['order_date'])

# Agrupar revenue por mes
revenue_mensual = df.resample('ME', on='order_date')['order_value'].sum().reset_index()
revenue_mensual.columns = ['ds', 'y']

print(f"Rango de fechas: {revenue_mensual['ds'].min()} → {revenue_mensual['ds'].max()}")
print(f"Meses disponibles: {len(revenue_mensual)}")
revenue_mensual

Rango de fechas: 2016-09-30 00:00:00 → 2018-08-31 00:00:00
Meses disponibles: 24


,ds,y
0,2016-09-30,143.46
1,2016-10-31,48492.16
2,2016-11-30,0.00
3,2016-12-31,39.24
4,2017-01-31,139783.66
5,2017-02-28,285104.63
6,2017-03-31,444185.91
7,2017-04-30,422099.52
8,2017-05-31,617509.14
9,2017-06-30,532362.02


In [4]:
# Entrenar modelo Prophet 
modelo = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='additive',
    changepoint_prior_scale=0.05
)

modelo.fit(revenue_mensual)

# Proyectar 6 meses hacia adelante
futuro = modelo.make_future_dataframe(periods=6, freq='MS')
forecast = modelo.predict(futuro)

# Asegurar que no haya valores negativos
forecast['yhat'] = forecast['yhat'].clip(lower=0)
forecast['yhat_lower'] = forecast['yhat_lower'].clip(lower=0)
forecast['yhat_upper'] = forecast['yhat_upper'].clip(lower=0)

print("Forecast generado ✅")
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(8)

18:43:48 - cmdstanpy - INFO - Chain [1] start processing


18:43:48 - cmdstanpy - INFO - Chain [1] done processing


Forecast generado ✅


,ds,yhat,yhat_lower,yhat_upper
22,2018-07-31,1.297578e+06,1.117556e+06,1.481099e+06
23,2018-08-31,1.356746e+06,1.178404e+06,1.524634e+06
24,2018-09-01,1.358655e+06,1.175081e+06,1.533908e+06
25,2018-10-01,1.415915e+06,1.232216e+06,1.603996e+06
26,2018-11-01,1.475084e+06,1.295637e+06,1.663401e+06
27,2018-12-01,1.532344e+06,1.349356e+06,1.722399e+06
28,2019-01-01,1.591513e+06,1.412280e+06,1.772430e+06
29,2019-02-01,1.650682e+06,1.472956e+06,1.828671e+06


In [5]:
# Visualización con Plotly
fig = go.Figure()

# Datos históricos
fig.add_trace(go.Scatter(
    x=revenue_mensual['ds'],
    y=revenue_mensual['y'],
    name='Revenue histórico',
    line=dict(color='#2196F3', width=2),
    mode='lines+markers'
))

# Predicción
fig.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat'],
    name='Forecast',
    line=dict(color='#FF9800', width=2, dash='dash')
))

# Intervalo de confianza
fig.add_trace(go.Scatter(
    x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
    y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(255,152,0,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Intervalo de confianza'
))

fig.update_layout(
    title='Forecast de Revenue Mensual — Olist 2016-2019',
    xaxis_title='Fecha',
    yaxis_title='Revenue (BRL)',
    template='plotly_dark',
    hovermode='x unified',
    height=500
)

fig.show()

In [6]:
# Resumen ejecutivo del forecast
historico_ultimo = revenue_mensual['y'].iloc[-1]
forecast_6m = forecast[forecast['ds'] > revenue_mensual['ds'].max()]['yhat'].sum()
crecimiento = ((forecast['yhat'].iloc[-1] - historico_ultimo) / historico_ultimo) * 100

print("=" * 50)
print("RESUMEN EJECUTIVO — FORECAST DE REVENUE")
print("=" * 50)
print(f"Revenue último mes histórico:  R$ {historico_ultimo:,.0f}")
print(f"Revenue proyectado (6 meses):  R$ {forecast_6m:,.0f}")
print(f"Crecimiento proyectado:        {crecimiento:.1f}%")
print(f"Revenue proyectado Feb 2019:   R$ {forecast['yhat'].iloc[-1]:,.0f}")
print("=" * 50)
print("\n⚠️  Nota: Proyección basada en tendencia histórica 2016-2018.")
print("El modelo asume continuidad del crecimiento observado.")

RESUMEN EJECUTIVO — FORECAST DE REVENUE
Revenue último mes histórico:  R$ 1,043,374
Revenue proyectado (6 meses):  R$ 9,024,193
Crecimiento proyectado:        58.2%
Revenue proyectado Feb 2019:   R$ 1,650,682

⚠️  Nota: Proyección basada en tendencia histórica 2016-2018.
El modelo asume continuidad del crecimiento observado.
